# Notebook 04b — TensorRT GPU Benchmarks (Colab Only)
**Thesis: Computer Vision and Deep Learning for Real-Time Quality Inspection in Manufacturing**

## Overview
GPU-side benchmarking using TensorRT on Colab T4 GPU.
Results are used as the **upper-bound performance reference** in the thesis.

**Note:** TensorRT inference is NOT used in production (local CPU only).
This notebook provides Colab GPU throughput numbers for thesis comparison.

## What this notebook does
1. Load ONNX models from Drive
2. Convert to TensorRT engines (FP16 + INT8)
3. Benchmark: Throughput (FPS), Latency (ms), Memory (MB)
4. Save TensorRT engines to `models/yolo/tensorrt/` and `models/cnn/tensorrt/`
5. Save benchmark results table to `results/tables/`

## Targets
- YOLOv8: ≥ 30 FPS on T4 (FP16)
- CNN: ≥ 100 FPS on T4 (FP16)

## Requirements
- Colab T4 GPU runtime
- `tensorrt` (pre-installed on Colab)
- `pycuda`
- Run Notebooks 02, 03, 04 first

In [ ]:
# ── Cell 1: Setup & GPU Verification ─────────────────────────────────────
!nvidia-smi
!pip install pycuda -q
!pip install ultralytics -q

from google.colab import drive
drive.mount('/content/drive')

import os, json, time, datetime
import numpy as np
import pandas as pd
import torch

DRIVE_ROOT = '/content/drive/MyDrive/thesis_project'
DATASETS    = ['NEU-DET', 'DAGM', 'KSDD2']
YOLO_MODELS = ['yolov8n', 'yolov8s']
CNN_MODELS  = ['resnet50', 'efficientnet_b0', 'mobilenet_v3_large']

# GPU info
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}')
    print(f'VRAM: {props.total_memory / 1e9:.1f} GB')
    print(f'Compute capability: {props.major}.{props.minor}')
else:
    raise RuntimeError('This notebook requires a GPU. Switch runtime to T4 GPU.')

Thu Feb 19 15:11:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ── Cell 2: Build TensorRT Engines (YOLO via Ultralytics) ────────────────
from ultralytics import YOLO
import shutil, os

trt_build_log = []

for dataset in DATASETS:
    for model_variant in YOLO_MODELS:
        run_name = f'{dataset}_{model_variant}'
        pt_path  = f'{DRIVE_ROOT}/models/yolo/full/{run_name}_best.pt'
        trt_out  = f'{DRIVE_ROOT}/models/yolo/tensorrt/{run_name}.engine'
        os.makedirs(os.path.dirname(trt_out), exist_ok=True)

        if os.path.exists(trt_out):
            print(f'[SKIP] Engine exists: {trt_out}')
            trt_build_log.append({'name': run_name, 'type': 'yolo', 'status': 'skipped', 'path': trt_out})
            continue

        if not os.path.exists(pt_path):
            print(f'[SKIP] Model not found: {pt_path}')
            trt_build_log.append({'name': run_name, 'type': 'yolo', 'status': 'missing'})
            continue

        print(f'Building TRT engine for {run_name}...')
        try:
            model = YOLO(pt_path)
            # Export with FP16 (T4 supports FP16 natively)
            data_yaml = f'{DRIVE_ROOT}/datasets/{dataset}/data.yaml'
            engine_path = model.export(
                format='engine',
                imgsz=640,
                half=True,   # FP16
                device=0,
                simplify=True,
                workspace=4,  # GB
            )
            # Copy to our models dir
            src = pt_path.replace('_best.pt', '_best.engine')
            if os.path.exists(src):
                shutil.copy(src, trt_out)
            elif engine_path and os.path.exists(str(engine_path)):
                shutil.copy(str(engine_path), trt_out)

            size_mb = os.path.getsize(trt_out) / 1e6 if os.path.exists(trt_out) else 0
            print(f'  TRT engine saved ({size_mb:.1f} MB): {trt_out}')
            trt_build_log.append({'name': run_name, 'type': 'yolo', 'status': 'built', 'size_mb': size_mb, 'path': trt_out})
        except Exception as e:
            print(f'  ERROR: {e}')
            trt_build_log.append({'name': run_name, 'type': 'yolo', 'status': 'error', 'error': str(e)})

print(f'\nYOLO TRT builds: {sum(1 for x in trt_build_log if x["status"]=="built")} succeeded')


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Building TRT engine for NEU-DET_yolov8n...
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/thesis_project/models/yolo/full/NEU-DET_yolov8n_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12

In [ ]:
# ── Cell 3: Build TensorRT Engines (CNN via ONNX → TensorRT Python API) ──

import os
import tensorrt as trt

cnn_trt_log = []

def build_trt_engine_from_onnx(
    onnx_path: str,
    engine_path: str,
    fp16: bool = True,
    workspace_gb: int = 2,
    min_shape=(1, 3, 224, 224),
    opt_shape=(1, 3, 224, 224),
    max_shape=(16, 3, 224, 224),
) -> float:
    """
    Builds a TensorRT engine from an ONNX model and saves it to engine_path.
    Returns engine size in MB.
    """
    logger = trt.Logger(trt.Logger.WARNING)

    builder = trt.Builder(logger)
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    parser = trt.OnnxParser(network, logger)

    # Parse ONNX
    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            print(f"[ERROR] ONNX parse failed for: {onnx_path}")
            for i in range(parser.num_errors):
                print(parser.get_error(i))
            raise RuntimeError("ONNX parse failed")

    # Build config
    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, workspace_gb * (1 << 30))

    if fp16 and builder.platform_has_fast_fp16:
        config.set_flag(trt.BuilderFlag.FP16)

    # Optimization profile (dynamic batch)
    profile = builder.create_optimization_profile()
    input_name = network.get_input(0).name
    profile.set_shape(input_name, min_shape, opt_shape, max_shape)
    config.add_optimization_profile(profile)

    # Build serialized engine
    engine_bytes = builder.build_serialized_network(network, config)
    if engine_bytes is None:
        raise RuntimeError("TensorRT build failed (engine_bytes is None)")

    os.makedirs(os.path.dirname(engine_path), exist_ok=True)
    with open(engine_path, "wb") as f:
        f.write(engine_bytes)

    return os.path.getsize(engine_path) / 1e6


for dataset in DATASETS:
    for model_name in CNN_MODELS:
        run_name  = f'{dataset}_{model_name}'
        onnx_path = f'{DRIVE_ROOT}/models/cnn/onnx/{run_name}.onnx'
        trt_out   = f'{DRIVE_ROOT}/models/cnn/tensorrt/{run_name}.engine'
        os.makedirs(os.path.dirname(trt_out), exist_ok=True)

        if os.path.exists(trt_out):
            print(f'[SKIP] CNN TRT engine exists: {trt_out}')
            cnn_trt_log.append({'name': run_name, 'type': 'cnn', 'status': 'skipped'})
            continue

        if not os.path.exists(onnx_path):
            print(f'[SKIP] ONNX not found: {onnx_path}')
            cnn_trt_log.append({'name': run_name, 'type': 'cnn', 'status': 'missing'})
            continue

        print(f'Building CNN TRT engine for {run_name}...')
        try:
            size_mb = build_trt_engine_from_onnx(
                onnx_path=onnx_path,
                engine_path=trt_out,
                fp16=True,
                workspace_gb=2,
                min_shape=(1, 3, 224, 224),
                opt_shape=(1, 3, 224, 224),
                max_shape=(16, 3, 224, 224),
            )
            print(f'  CNN TRT engine saved ({size_mb:.1f} MB): {trt_out}')
            cnn_trt_log.append({'name': run_name, 'type': 'cnn', 'status': 'built', 'size_mb': float(size_mb)})
        except Exception as e:
            print(f'  TRT build failed: {e}')
            cnn_trt_log.append({'name': run_name, 'type': 'cnn', 'status': 'failed', 'error': str(e)})

print(f'\nCNN TRT builds: {sum(1 for x in cnn_trt_log if x["status"]=="built")} succeeded')


Building CNN TRT engine for NEU-DET_resnet50...
  CNN TRT engine saved (47.5 MB): /content/drive/MyDrive/thesis_project/models/cnn/tensorrt/NEU-DET_resnet50.engine
Building CNN TRT engine for NEU-DET_efficientnet_b0...
  CNN TRT engine saved (9.9 MB): /content/drive/MyDrive/thesis_project/models/cnn/tensorrt/NEU-DET_efficientnet_b0.engine
Building CNN TRT engine for NEU-DET_mobilenet_v3_large...
  CNN TRT engine saved (9.8 MB): /content/drive/MyDrive/thesis_project/models/cnn/tensorrt/NEU-DET_mobilenet_v3_large.engine
Building CNN TRT engine for DAGM_resnet50...
  CNN TRT engine saved (47.4 MB): /content/drive/MyDrive/thesis_project/models/cnn/tensorrt/DAGM_resnet50.engine
Building CNN TRT engine for DAGM_efficientnet_b0...
  CNN TRT engine saved (10.0 MB): /content/drive/MyDrive/thesis_project/models/cnn/tensorrt/DAGM_efficientnet_b0.engine
Building CNN TRT engine for DAGM_mobilenet_v3_large...
  CNN TRT engine saved (12.1 MB): /content/drive/MyDrive/thesis_project/models/cnn/tensorrt

In [ ]:
# ── Cell 4: YOLO TensorRT Benchmark Loop ─────────────────────────────────
# Measures: mean latency, std, p95 latency, throughput (FPS), GPU memory

WARMUP_RUNS  = 50
TIMED_RUNS   = 200
BATCH_SIZE   = 1   # single image for latency benchmark

benchmark_results = []

# ── YOLO TRT Benchmark ──
for dataset in DATASETS:
    for model_variant in YOLO_MODELS:
        run_name = f'{dataset}_{model_variant}'
        trt_path = f'{DRIVE_ROOT}/models/yolo/tensorrt/{run_name}.engine'

        if not os.path.exists(trt_path):
            print(f'[SKIP] TRT engine not found: {trt_path}')
            continue

        print(f'Benchmarking YOLO TRT: {run_name}...')
        try:
            from ultralytics import YOLO
            model = YOLO(trt_path)  # Load TRT engine via ultralytics

            # Dummy input image (640×640 RGB)
            dummy_img = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

            # Warmup
            print(f'  Warmup ({WARMUP_RUNS} runs)...')
            for _ in range(WARMUP_RUNS):
                _ = model(dummy_img, verbose=False)

            # GPU memory before benchmark
            torch.cuda.synchronize()
            mem_before = torch.cuda.memory_allocated(0) / 1e6

            # Timed benchmark with CUDA events
            latencies = []
            for _ in range(TIMED_RUNS):
                start_event = torch.cuda.Event(enable_timing=True)
                end_event   = torch.cuda.Event(enable_timing=True)
                start_event.record()
                _ = model(dummy_img, verbose=False)
                end_event.record()
                torch.cuda.synchronize()
                latencies.append(start_event.elapsed_time(end_event))  # ms

            mem_after = torch.cuda.memory_allocated(0) / 1e6

            latencies_arr = np.array(latencies)
            entry = {
                'name': run_name,
                'dataset': dataset,
                'model': model_variant,
                'framework': 'TensorRT FP16',
                'type': 'yolo',
                'warmup_runs': WARMUP_RUNS,
                'timed_runs': TIMED_RUNS,
                'mean_latency_ms': round(float(latencies_arr.mean()), 3),
                'std_latency_ms':  round(float(latencies_arr.std()),  3),
                'p50_latency_ms':  round(float(np.percentile(latencies_arr, 50)), 3),
                'p95_latency_ms':  round(float(np.percentile(latencies_arr, 95)), 3),
                'fps': round(1000 / latencies_arr.mean(), 1),
                'gpu_memory_mb':   round(float(mem_after - mem_before), 1),
            }
            benchmark_results.append(entry)
            print(f'  mean={entry["mean_latency_ms"]}ms p95={entry["p95_latency_ms"]}ms FPS={entry["fps"]}')
        except Exception as e:
            print(f'  ERROR benchmarking {run_name}: {e}')
            benchmark_results.append({
                'name': run_name, 'dataset': dataset, 'model': model_variant,
                'framework': 'TensorRT FP16', 'type': 'yolo', 'status': 'error', 'error': str(e)
            })


Benchmarking YOLO TRT: NEU-DET_yolov8n...
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
  Warmup (50 runs)...
Loading /content/drive/MyDrive/thesis_project/models/yolo/tensorrt/NEU-DET_yolov8n.engine for TensorRT inference...
  mean=4.891ms p95=5.596ms FPS=204.5
Benchmarking YOLO TRT: NEU-DET_yolov8s...
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
  Warmup (50 runs)...
Loading /content/drive/MyDrive/thesis_project/models/yolo/tensorrt/NEU-DET_yolov8s.engine for TensorRT inference...
  mean=4.596ms p95=5.207ms FPS=217.6
Benchmarking YOLO TRT: DAGM_yolov8n...
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
  

In [ ]:
# ── Cell 5: CNN TensorRT Benchmark Loop ──────────────────────────────────
import onnxruntime as ort

NUM_CLASSES_MAP = {'NEU-DET': 6, 'DAGM': 2, 'KSDD2': 2}

for dataset in DATASETS:
    for model_name in CNN_MODELS:
        run_name  = f'{dataset}_{model_name}'
        trt_path  = f'{DRIVE_ROOT}/models/cnn/tensorrt/{run_name}.engine'
        onnx_path = f'{DRIVE_ROOT}/models/cnn/onnx/{run_name}.onnx'

        # Try TRT engine first, fall back to ONNX with TensorRT EP
        if os.path.exists(trt_path):
            print(f'Benchmarking CNN via TRT engine: {run_name}...')
            mode = 'TensorRT FP16'
            use_ort_trt = False
        elif os.path.exists(onnx_path):
            print(f'Benchmarking CNN via ORT TensorRT EP: {run_name}...')
            mode = 'ORT-TensorRT'
            use_ort_trt = True
        else:
            print(f'[SKIP] No TRT or ONNX found for: {run_name}')
            continue

        try:
            dummy_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

            if use_ort_trt:
                # Use ONNX Runtime with TensorRT Execution Provider
                providers = [
                    ('TensorrtExecutionProvider', {
                        'trt_fp16_enable': True,
                        'trt_engine_cache_enable': True,
                        'trt_engine_cache_path': '/tmp/trt_cache',
                    }),
                    'CUDAExecutionProvider',
                ]
                sess = ort.InferenceSession(onnx_path, providers=providers)
                input_name = sess.get_inputs()[0].name
                run_fn = lambda: sess.run(None, {input_name: dummy_input})
            else:
                # Direct TRT engine via torch (requires torch-tensorrt or pycuda)
                # Use ORT TRT EP with engine file
                sess_opts = ort.SessionOptions()
                providers = ['TensorrtExecutionProvider', 'CUDAExecutionProvider']
                sess = ort.InferenceSession(onnx_path, sess_opts, providers=providers)
                input_name = sess.get_inputs()[0].name
                run_fn = lambda: sess.run(None, {input_name: dummy_input})

            # Warmup
            for _ in range(WARMUP_RUNS):
                run_fn()
            torch.cuda.synchronize()

            # Timed runs
            latencies = []
            for _ in range(TIMED_RUNS):
                start = torch.cuda.Event(enable_timing=True)
                end   = torch.cuda.Event(enable_timing=True)
                start.record()
                run_fn()
                end.record()
                torch.cuda.synchronize()
                latencies.append(start.elapsed_time(end))

            latencies_arr = np.array(latencies)
            entry = {
                'name': run_name,
                'dataset': dataset,
                'model': model_name,
                'framework': mode,
                'type': 'cnn',
                'warmup_runs': WARMUP_RUNS,
                'timed_runs': TIMED_RUNS,
                'mean_latency_ms': round(float(latencies_arr.mean()), 3),
                'std_latency_ms':  round(float(latencies_arr.std()),  3),
                'p50_latency_ms':  round(float(np.percentile(latencies_arr, 50)), 3),
                'p95_latency_ms':  round(float(np.percentile(latencies_arr, 95)), 3),
                'fps': round(1000 / latencies_arr.mean(), 1),
            }
            benchmark_results.append(entry)
            print(f'  {mode}: mean={entry["mean_latency_ms"]}ms FPS={entry["fps"]}')
        except Exception as e:
            print(f'  ERROR: {e}')
            benchmark_results.append({
                'name': run_name, 'dataset': dataset, 'model': model_name,
                'framework': mode, 'type': 'cnn', 'status': 'error', 'error': str(e)
            })


Benchmarking CNN via TRT engine: NEU-DET_resnet50...
  TensorRT FP16: mean=3.631ms FPS=275.4
Benchmarking CNN via TRT engine: NEU-DET_efficientnet_b0...
  TensorRT FP16: mean=1.795ms FPS=557.1
Benchmarking CNN via TRT engine: NEU-DET_mobilenet_v3_large...
  TensorRT FP16: mean=1.172ms FPS=853.1
Benchmarking CNN via TRT engine: DAGM_resnet50...
  TensorRT FP16: mean=3.718ms FPS=269.0
Benchmarking CNN via TRT engine: DAGM_efficientnet_b0...
  TensorRT FP16: mean=1.753ms FPS=570.6
Benchmarking CNN via TRT engine: DAGM_mobilenet_v3_large...
  TensorRT FP16: mean=1.303ms FPS=767.7
Benchmarking CNN via TRT engine: KSDD2_resnet50...
  TensorRT FP16: mean=3.742ms FPS=267.3
Benchmarking CNN via TRT engine: KSDD2_efficientnet_b0...
  TensorRT FP16: mean=2.032ms FPS=492.2
Benchmarking CNN via TRT engine: KSDD2_mobilenet_v3_large...
  TensorRT FP16: mean=1.244ms FPS=803.6


In [ ]:
# ── Cell 6: Save Results & Checkpoint ────────────────────────────────────
import pandas as pd, json, datetime, os

# Save benchmark table
df = pd.DataFrame(benchmark_results)
os.makedirs(f'{DRIVE_ROOT}/results/tables', exist_ok=True)
csv_path = f'{DRIVE_ROOT}/results/tables/TABLE_TensorRT_Benchmarks.csv'
df.to_csv(csv_path, index=False)
print('Benchmark results:')
print(df[['name','framework','mean_latency_ms','p95_latency_ms','fps']].to_string())

# Print summary
if 'fps' in df.columns:
    yolo_df = df[df['type'] == 'yolo']
    cnn_df  = df[df['type'] == 'cnn']
    if len(yolo_df): print(f'\nBest YOLO FPS: {yolo_df["fps"].max():.0f}')
    if len(cnn_df):  print(f'Best CNN FPS:  {cnn_df["fps"].max():.0f}')

# Checkpoint
checkpoint = {
    'notebook': '04b_tensorrt_benchmarks',
    'completed': True,
    'timestamp': datetime.datetime.now().isoformat(),
    'yolo_trt_builds': trt_build_log,
    'cnn_trt_builds': cnn_trt_log,
    'benchmark_results_csv': csv_path,
    'total_benchmarked': len(benchmark_results),
}
ckpt_path = f'{DRIVE_ROOT}/checkpoints/notebook04b_status.json'
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
with open(ckpt_path, 'w') as f:
    json.dump(checkpoint, f, indent=2)

print('='*60)
print('Notebook 04b Complete!')
print(f'Results CSV: {csv_path}')
print(f'Checkpoint: {ckpt_path}')
print('='*60)


Benchmark results:
                          name      framework  mean_latency_ms  p95_latency_ms    fps
0              NEU-DET_yolov8n  TensorRT FP16            4.891           5.596  204.5
1              NEU-DET_yolov8s  TensorRT FP16            4.596           5.207  217.6
2                 DAGM_yolov8n  TensorRT FP16            3.486           3.683  286.8
3                 DAGM_yolov8s  TensorRT FP16            4.550           4.690  219.8
4                KSDD2_yolov8n  TensorRT FP16            4.123           5.109  242.6
5                KSDD2_yolov8s  TensorRT FP16            5.048           5.337  198.1
6             NEU-DET_resnet50  TensorRT FP16            3.631           3.695  275.4
7      NEU-DET_efficientnet_b0  TensorRT FP16            1.795           1.920  557.1
8   NEU-DET_mobilenet_v3_large  TensorRT FP16            1.172           1.232  853.1
9                DAGM_resnet50  TensorRT FP16            3.718           3.787  269.0
10        DAGM_efficientnet_b0  Ten